# Lezione 5: Ottimizzazione II — tecniche avanzate e regolarizzazione

In questa lezione riprendi la discesa del gradiente che hai studiato nella lezione precedente e ne risolvi il punto debole: la lentezza su paesaggi di perdita mal condizionati. Esegui ogni cella di codice man mano che procedi nella spiegazione; questo quaderno è pensato per essere proiettato ed eseguito dal vivo durante la sessione.

**Obiettivi di apprendimento**

Al termine di questa lezione sarai in grado di:

- Spiegare perché la discesa del gradiente pura oscilla nelle valli strette
- Implementare l'aggiornamento con momento e confrontare la sua traiettoria con quella della discesa del gradiente pura
- Descrivere che cosa apporta la matrice Hessiana e perché invertirla è costoso nelle reti con molti parametri
- Confrontare SGD, SGD con momento e Adam sullo stesso problema, in PyTorch
- Applicare dropout e weight decay come tecniche di regolarizzazione in un modello PyTorch

## Il problema delle valli strette

Ricorda la lezione precedente: quando il paesaggio di perdita ha una direzione molto più curvata di un'altra, la discesa del gradiente rimbalza tra le due pareti della valle e avanza molto poco verso l'uscita. Questo zigzag compare perché il gradiente punta con forza verso i fianchi e solo un po' in avanti, cioè nella direzione in cui vuoi davvero andare.

Due sintomi dello stesso problema:

- Se riduci il tasso di apprendimento per non uscire dalla valle, avanzi in modo sicuro ma lento
- Se lo aumenti per avanzare più in fretta, oscilli ancora di più o diverghi

Oggi risolvi questo problema con due idee: il momento e, più avanti, l'informazione sulla curvatura.

## Momento: una palla che accumula inerzia

Immagina di lasciar rotolare una palla pesante nella stessa valle, invece di camminare guardando solo la pendenza sotto i tuoi piedi. La palla non reagisce a ogni irregolarità del terreno: accumula la velocità che aveva già, cioè possiede inerzia.

Le forze che spingono la palla verso un fianco e verso l'altro cambiano segno e tendono a cancellarsi. La forza che la spinge lungo il letto della valle, verso l'uscita, punta sempre nella stessa direzione e si rafforza passo dopo passo. Il risultato: la palla rotola veloce lungo la valle e rimbalza appena tra le pareti.

La regola di aggiornamento con momento:

v_t = β · v_{t−1} + η · ∇L(θ_{t−1})

θ_t = θ_{t−1} − v_t

Il coefficiente β (tipicamente tra 0,9 e 0,99) controlla quanto la velocità "ricorda" i gradienti precedenti. Confronta ora, a mano, la traiettoria delle due versioni su una valle stretta.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Función de pérdida con un valle estrecho: la dirección x está mucho
# más curvada que la dirección y, igual que en la analogía del valle
def perdida(x, y):
    return 25 * x**2 + y**2

def gradiente(x, y):
    return np.array([50 * x, 2 * y])

punto_inicial = np.array([-4.0, 8.0])
n_pasos = 60
tasa_aprendizaje = 0.018

# --- Descenso de gradiente puro ---
theta = punto_inicial.copy()
trayectoria_gd = [theta.copy()]
for _ in range(n_pasos):
    theta = theta - tasa_aprendizaje * gradiente(*theta)
    trayectoria_gd.append(theta.copy())
trayectoria_gd = np.array(trayectoria_gd)

# --- Descenso de gradiente con momento, implementado a mano ---
beta = 0.9
theta = punto_inicial.copy()
v = np.zeros(2)
trayectoria_momento = [theta.copy()]
for _ in range(n_pasos):
    v = beta * v + tasa_aprendizaje * gradiente(*theta)
    theta = theta - v
    trayectoria_momento.append(theta.copy())
trayectoria_momento = np.array(trayectoria_momento)

# --- Gráfico: curvas de nivel de la pérdida + las dos trayectorias ---
x_malla, y_malla = np.meshgrid(np.linspace(-5, 2, 200), np.linspace(-9, 9, 200))
z_malla = perdida(x_malla, y_malla)

plt.figure(figsize=(8, 6))
plt.contour(x_malla, y_malla, z_malla, levels=30, cmap="Greys", alpha=0.6)
plt.plot(trayectoria_gd[:, 0], trayectoria_gd[:, 1], "o-", markersize=3,
         label="Descenso de gradiente puro", color="tab:red")
plt.plot(trayectoria_momento[:, 0], trayectoria_momento[:, 1], "o-", markersize=3,
         label="Descenso de gradiente con momento", color="tab:blue")
plt.scatter([0], [0], color="black", marker="*", s=150, zorder=5, label="Mínimo")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Trayectoria en un valle estrecho: con y sin momento")
plt.legend()
plt.show()

print(f"Distancia al mínimo tras {n_pasos} pasos:")
print(f"  Sin momento: {np.linalg.norm(trayectoria_gd[-1]):.4f}")
print(f"  Con momento: {np.linalg.norm(trayectoria_momento[-1]):.4f}")

## Metodi del secondo ordine e perché sono costosi

Il momento usa ancora solo il gradiente, cioè la pendenza locale. I metodi del secondo ordine usano anche la curvatura del terreno, catturata dalla matrice Hessiana, H, che raccoglie tutte le derivate parziali del secondo ordine della perdita rispetto ai parametri:

H_ij = ∂²L(θ) / ∂θ_i ∂θ_j

Con questa informazione, il metodo di Newton calcola l'aggiornamento:

θ_{t+1} = θ_t − η · H⁻¹ · ∇L(θ_t)

In una direzione molto curvata, H⁻¹ riduce il passo; in una direzione poco curvata, lo amplia. Nella valle stretta, Newton "sa" in anticipo che un fianco è più ripido dell'altro e regola il passo in un solo aggiornamento, invece di avere bisogno di centinaia di piccoli passi.

Il problema è il costo. Per un modello con n parametri, H è una matrice n × n:

- Memorizzarla per intero costa O(n²)
- Invertirla costa O(n³)

Una rete modesta ha già diversi milioni di parametri: elevare quella cifra al cubo dà un numero di operazioni inaccessibile, anche sull'hardware più potente disponibile oggi. Confrontalo con il costo di un aggiornamento della discesa del gradiente pura, che è O(n). Questo divario spiega perché quasi nessuna rete profonda viene addestrata con il metodo di Newton puro. Esistono alternative che riducono il costo rinunciando a parte dell'informazione sulla curvatura: i metodi quasi-Newton, come BFGS (Broyden, Fletcher, Goldfarb e Shanno), e le approssimazioni diagonali a blocchi o di Kronecker, note come K-FAC (curvatura approssimata fattorizzata di Kronecker).

La cella successiva misura, con NumPy, quanto cresce in pratica il tempo necessario per invertire una matrice quadrata al crescere della sua dimensione.

In [ ]:
import time
import numpy as np

tamanos = [200, 400, 800, 1600]
tiempos = []

for n in tamanos:
    matriz = np.random.randn(n, n)
    matriz = matriz @ matriz.T + n * np.eye(n)  # la hacemos invertible y bien condicionada

    inicio = time.time()
    np.linalg.inv(matriz)
    tiempos.append(time.time() - inicio)

for n, t in zip(tamanos, tiempos):
    print(f"n = {n:>5}  ->  {t:.4f} segundos")

print()
print("Al duplicar n, el tiempo debería multiplicarse por, aproximadamente, ocho (2³),")
print("porque el coste de invertir una matriz crece como O(n³).")

## SGD, Momento e Adam a confronto

Ora confronta i tre ottimizzatori su un problema con la stessa curvatura mal condizionata di prima, ma con una rete neurale al posto di un singolo punto. Addestri tre volte lo stesso perceptrone multistrato (MLP), partendo dagli stessi pesi iniziali, e cambi soltanto l'ottimizzatore: la discesa del gradiente stocastica (SGD) pura, SGD con momento (0,9) e Adam.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)

# --- Datos sintéticos con curvatura mal condicionada ---
# Una característica tiene mucha más varianza que la otra: esto genera
# un paisaje de pérdida en forma de valle estrecho, igual que en la analogía
n_muestras = 500
X = torch.randn(n_muestras, 2)
X[:, 0] *= 20.0

pesos_reales = torch.tensor([[3.0], [-2.0]])
y = X @ pesos_reales + 0.1 * torch.randn(n_muestras, 1)


def crear_mlp():
    """Perceptrón multicapa pequeño: dos capas ocultas de 16 neuronas.
    Fijamos la semilla para partir de los mismos pesos en las tres pruebas."""
    torch.manual_seed(42)
    return nn.Sequential(
        nn.Linear(2, 16),
        nn.ReLU(),
        nn.Linear(16, 16),
        nn.ReLU(),
        nn.Linear(16, 1),
    )


def entrenar(optimizador_fn, n_pasos=200):
    modelo = crear_mlp()
    perdida_fn = nn.MSELoss()  # error cuadrático medio (MSE, por sus siglas en inglés)
    optimizador = optimizador_fn(modelo.parameters())

    historial_perdida = []
    for _ in range(n_pasos):
        optimizador.zero_grad()
        pred = modelo(X)
        perdida = perdida_fn(pred, y)
        perdida.backward()
        optimizador.step()
        historial_perdida.append(perdida.item())

    return historial_perdida


tasa_aprendizaje = 0.001

resultados = {
    "SGD (sin momento)": entrenar(
        lambda params: torch.optim.SGD(params, lr=tasa_aprendizaje)
    ),
    "SGD con momento (0.9)": entrenar(
        lambda params: torch.optim.SGD(params, lr=tasa_aprendizaje, momentum=0.9)
    ),
    "Adam": entrenar(
        lambda params: torch.optim.Adam(params, lr=tasa_aprendizaje)
    ),
}

plt.figure(figsize=(8, 5))
for nombre, perdidas in resultados.items():
    plt.plot(perdidas, label=nombre)

plt.yscale("log")
plt.xlabel("Iteración")
plt.ylabel("Pérdida (MSE, escala log)")
plt.title("SGD, momento y Adam sobre el mismo MLP")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Regolarizzazione in pratica

L'ottimizzatore risolve la velocità di convergenza; la regolarizzazione risolve un problema diverso, il sovradattamento (*overfitting*), cioè il fatto che il modello memorizzi i dati di addestramento invece di generalizzare a dati nuovi. Due tecniche standard:

- **Dropout.** A ogni passo di addestramento disattiva casualmente una percentuale di neuroni (di solito tra il 20 % e il 50 %). Questo impedisce alla rete di dipendere troppo da neuroni specifici. In PyTorch si aggiunge con `torch.nn.Dropout`
- **Weight decay.** Aggiunge una penalità proporzionale all'entità dei pesi nella funzione di perdita, spingendo il modello verso soluzioni più semplici. In PyTorch si attiva con il parametro `weight_decay` di quasi tutti gli ottimizzatori

**Combinale** ogni volta che addestri un modello reale: ottimizzatore e regolarizzazione lavorano insieme, non in modo isolato.

In [ ]:
import torch.nn as nn
import torch.optim as optim

modelo_regularizado = nn.Sequential(
    nn.Linear(2, 32),
    nn.ReLU(),
    nn.Dropout(0.3),      # desactiva el 30 % de las neuronas en cada paso de entrenamiento
    nn.Linear(32, 32),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(32, 1),
)

optimizador_regularizado = optim.Adam(
    modelo_regularizado.parameters(),
    lr=0.001,
    weight_decay=1e-4,   # penaliza los pesos grandes dentro de la función de pérdida
)

# modelo_regularizado.train() activa el dropout; modelo_regularizado.eval() lo desactiva
modelo_regularizado.train()
salida_entrenamiento = modelo_regularizado(X)

modelo_regularizado.eval()
salida_evaluacion = modelo_regularizado(X)

print("Optimizador:", optimizador_regularizado)
print()
print("Salida en modo entrenamiento (primeras tres filas):")
print(salida_entrenamiento[:3])
print()
print("Salida en modo evaluación (primeras tres filas):")
print(salida_evaluacion[:3])

## Riepilogo della lezione

- La discesa del gradiente pura è lenta o oscillante su paesaggi di perdita mal condizionati, come le valli strette
- Il momento accumula una media mobile dei gradienti passati: attenua le oscillazioni e rafforza l'avanzamento nelle direzioni coerenti
- La matrice Hessiana fornisce informazione sulla curvatura, ma calcolarla e invertirla costa rispettivamente O(n²) e O(n³), un costo inaccessibile nelle reti con milioni di parametri
- Esistono approssimazioni (quasi-Newton, diagonali a blocchi, K-FAC) che riducono questo costo rinunciando a parte dell'informazione sulla curvatura
- Adam di solito converge più rapidamente di SGD e di SGD con momento su problemi mal condizionati, perché adatta anche il tasso di apprendimento per ciascun parametro
- Dropout e weight decay sono tecniche di regolarizzazione che migliorano la generalizzazione del modello, non la velocità di convergenza

**Apri ora il quaderno di pratica `../notebooks_practica/clase05_practica_optimizacion_avanzada.ipynb` e metti in pratica ciò che hai appena visto.**